# PS S6E6 - 019 Blend add018 XGB check

Experiment: `exp_20260607_019_blend_add018_xgb_check`

Purpose:
- Add `018 shared003 XGBoost as-is` to the current RealMLP branch.
- Check whether weak-single XGB still adds complementary signal to 017 current best.
- Compare 014 / 015 / 016 / 017 / 018 by individual CV, pairwise correlation, disagreement, error overlap, and safe blends.
- Save only the best safe blend OOF/pred `.npy` files.

Current references:
- 017 bias-search-on-015-RealMLP: CV `0.9683022653955233`, Public LB `0.96985`
- 015 shared001 RealMLP as-is: CV `0.9681693449222352`, Public LB `0.96977`
- 016 add015 blend: CV `0.968246781866073`, Public LB `0.96948`
- 018 shared003 XGB as-is: CV `0.965207986418628`, Public LB `0.96578`

Important:
- Input `.npy` files are read from `/kaggle/input/datasets/mizushimatoshihiko/npy-files`.
- Safe final candidates are only `avg`, `hc_nonnegative_raw`, and `nm_softmax_raw`.
- LogReg / Ridge / ElasticNet rows are in-sample meta screening only, not final candidates.

In [1]:
import os
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260607_019_blend_add018_xgb_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {
        "key": "014_blend_bias",
        "exp_id": "exp_20260605_014_bias_search_on_013_best_blend",
        "family": "Blend",
        "role": "old_blend_bias_reference",
        "oof": "oof_exp_20260605_014_bias_search_on_013_best_blend_proba.npy",
        "pred": "pred_exp_20260605_014_bias_search_on_013_best_blend_proba.npy",
        "cv": 0.9660085788215015,
        "public_lb": 0.96703,
    },
    {
        "key": "015_realmlp_shared001",
        "exp_id": "exp_20260605_015_shared001_realmlp_as_is",
        "family": "RealMLP",
        "role": "previous_best_single_realmlp",
        "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "cv": 0.9681693449222352,
        "public_lb": 0.96977,
    },
    {
        "key": "016_blend_best",
        "exp_id": "exp_20260605_016_blend_add015_realmlp_check",
        "family": "Blend",
        "role": "hold_cv_up_lb_down",
        "oof": "oof_exp_20260605_016_blend_add015_realmlp_check_best_blend_proba.npy",
        "pred": "pred_exp_20260605_016_blend_add015_realmlp_check_best_blend_proba.npy",
        "cv": 0.968246781866073,
        "public_lb": 0.96948,
    },
    {
        "key": "017_realmlp_bias",
        "exp_id": "exp_20260606_017_bias_search_on_015_realmlp",
        "family": "RealMLP",
        "role": "current_best_realmlp_bias",
        "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "cv": 0.9683022653955233,
        "public_lb": 0.96985,
    },
    {
        "key": "018_xgb_shared003",
        "exp_id": "exp_20260606_018_shared003_xgb_as_is",
        "family": "XGBoost",
        "role": "xgb_hold_for_blend_check",
        "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "cv": 0.965207986418628,
        "public_lb": 0.96578,
    },
]

BLEND_SETS = {
    "A_017_only": ["017_realmlp_bias"],
    "B_015_only": ["015_realmlp_shared001"],
    "C_018_only": ["018_xgb_shared003"],
    "D_017_018": ["017_realmlp_bias", "018_xgb_shared003"],
    "E_015_018": ["015_realmlp_shared001", "018_xgb_shared003"],
    "F_014_017_018": ["014_blend_bias", "017_realmlp_bias", "018_xgb_shared003"],
    "G_015_017_018": ["015_realmlp_shared001", "017_realmlp_bias", "018_xgb_shared003"],
    "H_016_018": ["016_blend_best", "018_xgb_shared003"],
    "I_016_017_018": ["016_blend_best", "017_realmlp_bias", "018_xgb_shared003"],
    "J_014_015_017_018": ["014_blend_bias", "015_realmlp_shared001", "017_realmlp_bias", "018_xgb_shared003"],
    "K_014_015_016_017_018": ["014_blend_bias", "015_realmlp_shared001", "016_blend_best", "017_realmlp_bias", "018_xgb_shared003"],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)

EXP_ID: exp_20260607_019_blend_add018_xgb_check
OUTDIR: /kaggle/working/exp_20260607_019_blend_add018_xgb_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files


In [2]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def load_npy_from_dataset(filename):
    path = NPY_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing npy file: {path}\n"
            f"Add the output file to the npy-files Kaggle dataset or edit MODEL_SPECS."
        )
    return path

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape, n_rows, n_classes)
    assert np.isfinite(arr).all(), name
    row_sum = arr.sum(axis=1)
    assert np.allclose(row_sum, 1.0, atol=1e-4), (name, row_sum.min(), row_sum.max())
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats.append(p)
            mats.append(logit_transform(p))
        elif mode == "raw_rank_logit":
            mats.append(p)
            mats.append(rank_normalize_matrix(p))
            mats.append(logit_transform(p))
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    cur_score = balanced_accuracy_score(y_true, cur.argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]

    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score = cur_score
            best_w = w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w = cand_w / cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score = score
                        best_w = cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score = best_score
                w = best_w
                improved = True
                hist.append({"iter": len(hist), "score": float(cur_score), "weights": w.copy().tolist()})

    final = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    return w, final, float(cur_score), hist

def nm_optimize_weights(y_true, probas, maxiter=1500):
    n = len(probas)
    def unpack(theta):
        e = np.exp(theta - np.max(theta))
        return e / e.sum()
    def objective(theta):
        w = unpack(theta)
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return -balanced_accuracy_score(y_true, p.argmax(axis=1))
    res = minimize(objective, np.zeros(n), method="Nelder-Mead",
                   options={"maxiter": maxiter, "xatol": 1e-7, "fatol": 1e-10})
    w = unpack(res.x)
    p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    score = balanced_accuracy_score(y_true, p.argmax(axis=1))
    return w, p, float(score), res

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

In [3]:
for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)
if not NPY_DIR.exists():
    raise FileNotFoundError(NPY_DIR)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof = {}
pred = {}
resolved_specs = []

for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = load_npy_from_dataset(spec["oof"])
    pred_path = load_npy_from_dataset(spec["pred"])

    oof_arr = np.load(oof_path).astype(np.float32)
    pred_arr = np.load(pred_path).astype(np.float32)

    validate_proba(f"{key} oof", oof_arr, len(train), n_classes)
    validate_proba(f"{key} pred", pred_arr, len(test), n_classes)

    oof[key] = oof_arr
    pred[key] = pred_arr

    row = dict(spec)
    row["resolved_oof_path"] = str(oof_path)
    row["resolved_pred_path"] = str(pred_path)
    row["oof_shape"] = list(oof_arr.shape)
    row["pred_shape"] = list(pred_arr.shape)
    resolved_specs.append(row)

model_keys = [s["key"] for s in MODEL_SPECS]

print("train:", train.shape)
print("test :", test.shape)
print("class_names:", class_names)
display(pd.DataFrame(resolved_specs))

train: (577347, 12)
test : (247435, 11)
class_names: ['GALAXY', 'QSO', 'STAR']


,key,exp_id,family,role,oof,pred,cv,public_lb,resolved_oof_path,resolved_pred_path,oof_shape,pred_shape
0,014_blend_bias,exp_20260605_014_bias_search_on_013_best_blend,Blend,old_blend_bias_reference,oof_exp_20260605_014_bias_search_on_013_best_b...,pred_exp_20260605_014_bias_search_on_013_best_...,0.966009,0.96703,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
1,015_realmlp_shared001,exp_20260605_015_shared001_realmlp_as_is,RealMLP,previous_best_single_realmlp,oof_exp_20260605_015_shared001_realmlp_as_is_p...,pred_exp_20260605_015_shared001_realmlp_as_is_...,0.968169,0.96977,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
2,016_blend_best,exp_20260605_016_blend_add015_realmlp_check,Blend,hold_cv_up_lb_down,oof_exp_20260605_016_blend_add015_realmlp_chec...,pred_exp_20260605_016_blend_add015_realmlp_che...,0.968247,0.96948,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
3,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,current_best_realmlp_bias,oof_exp_20260606_017_bias_search_on_015_realml...,pred_exp_20260606_017_bias_search_on_015_realm...,0.968302,0.96985,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
4,018_xgb_shared003,exp_20260606_018_shared003_xgb_as_is,XGBoost,xgb_hold_for_blend_check,oof_exp_20260606_018_shared003_xgb_as_is_proba...,pred_exp_20260606_018_shared003_xgb_as_is_prob...,0.965208,0.96578,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"


In [4]:
rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)

    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
3,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,current_best_realmlp_bias,0.968302,0.96985,0.968302,0.0,0.959889,0.975867,0.969150
2,016_blend_best,exp_20260605_016_blend_add015_realmlp_check,Blend,hold_cv_up_lb_down,0.968247,0.96948,0.968247,0.0,0.964056,0.975329,0.965355
1,015_realmlp_shared001,exp_20260605_015_shared001_realmlp_as_is,RealMLP,previous_best_single_realmlp,0.968169,0.96977,0.968169,0.0,0.962234,0.976098,0.966177
0,014_blend_bias,exp_20260605_014_bias_search_on_013_best_blend,Blend,old_blend_bias_reference,0.966009,0.96703,0.966009,0.0,0.968062,0.970617,0.959347
4,018_xgb_shared003,exp_20260606_018_shared003_xgb_as_is,XGBoost,xgb_hold_for_blend_check,0.965208,0.96578,0.965208,0.0,0.966870,0.972043,0.956711


In [5]:
pair_rows = []
pred_labels = {k: proba_to_pred(oof[k]) for k in model_keys}
wrong_flags = {k: pred_labels[k] != y for k in model_keys}

for a, b in combinations(model_keys, 2):
    pa, pb = oof[a], oof[b]
    ya, yb = pred_labels[a], pred_labels[b]
    wrong_a, wrong_b = wrong_flags[a], wrong_flags[b]
    both = wrong_a & wrong_b
    either = wrong_a | wrong_b

    row = {
        "model_a": a,
        "model_b": b,
        "pearson_flat_proba": corr_pearson(flatten_proba(pa), flatten_proba(pb)),
        "spearman_flat_proba": corr_spearman(flatten_proba(pa), flatten_proba(pb)),
        "argmax_agreement": float((ya == yb).mean()),
        "argmax_disagreement": float((ya != yb).mean()),
        "wrong_rate_a": float(wrong_a.mean()),
        "wrong_rate_b": float(wrong_b.mean()),
        "both_wrong_rate": float(both.mean()),
        "either_wrong_rate": float(either.mean()),
        "error_jaccard": float(both.sum() / max(either.sum(), 1)),
        "a_wrong_b_correct_rate": float((wrong_a & ~wrong_b).mean()),
        "a_correct_b_wrong_rate": float((~wrong_a & wrong_b).mean()),
    }
    for ci, cls in enumerate(class_names):
        row[f"pearson_proba_{cls}"] = corr_pearson(pa[:, ci], pb[:, ci])
        row[f"spearman_proba_{cls}"] = corr_spearman(pa[:, ci], pb[:, ci])
    pair_rows.append(row)

pairwise_df = pd.DataFrame(pair_rows).sort_values("pearson_flat_proba")
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

pearson_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
spearman_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
argmax_agreement = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
error_jaccard = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)

for a in model_keys:
    for b in model_keys:
        pearson_flat.loc[a, b] = corr_pearson(flatten_proba(oof[a]), flatten_proba(oof[b]))
        spearman_flat.loc[a, b] = corr_spearman(flatten_proba(oof[a]), flatten_proba(oof[b]))
        argmax_agreement.loc[a, b] = float((pred_labels[a] == pred_labels[b]).mean())
        both = wrong_flags[a] & wrong_flags[b]
        either = wrong_flags[a] | wrong_flags[b]
        error_jaccard.loc[a, b] = float(both.sum() / max(either.sum(), 1))

display(pearson_flat)
display(spearman_flat)
display(argmax_agreement)
display(error_jaccard)

pearson_flat.to_csv(OUTDIR / "corr_matrix_pearson_flat_proba.csv")
spearman_flat.to_csv(OUTDIR / "corr_matrix_spearman_flat_proba.csv")
argmax_agreement.to_csv(OUTDIR / "matrix_argmax_agreement.csv")
error_jaccard.to_csv(OUTDIR / "matrix_error_jaccard.csv")

,model_a,model_b,pearson_flat_proba,spearman_flat_proba,argmax_agreement,argmax_disagreement,wrong_rate_a,wrong_rate_b,both_wrong_rate,either_wrong_rate,error_jaccard,a_wrong_b_correct_rate,a_correct_b_wrong_rate,pearson_proba_GALAXY,spearman_proba_GALAXY,pearson_proba_QSO,spearman_proba_QSO,pearson_proba_STAR,spearman_proba_STAR
9,017_realmlp_bias,018_xgb_shared003,0.990670,0.908650,0.981524,0.018476,0.035542,0.033536,0.025439,0.043639,0.582933,0.010103,0.008097,0.987587,0.939083,0.994756,0.857918,0.980882,0.780807
6,015_realmlp_shared001,018_xgb_shared003,0.991146,0.918772,0.982243,0.017757,0.034388,0.033536,0.025227,0.042697,0.590848,0.009161,0.008309,0.988216,0.939921,0.994776,0.854090,0.981740,0.778653
8,016_blend_best,018_xgb_shared003,0.992374,0.929872,0.983381,0.016619,0.033470,0.033536,0.025321,0.041686,0.607429,0.008149,0.008215,0.989789,0.947535,0.995553,0.862802,0.984041,0.791115
2,014_blend_bias,017_realmlp_bias,0.993651,0.904324,0.985206,0.014794,0.032668,0.035542,0.026857,0.041353,0.649466,0.005811,0.008685,0.991905,0.941093,0.995901,0.851459,0.987737,0.773884
3,014_blend_bias,018_xgb_shared003,0.993720,0.976021,0.984379,0.015621,0.032668,0.033536,0.025427,0.040778,0.623540,0.007242,0.008110,0.991644,0.967257,0.996035,0.894090,0.986881,0.925413
0,014_blend_bias,015_realmlp_shared001,0.994167,0.916308,0.985937,0.014063,0.032668,0.034388,0.026648,0.040409,0.659451,0.006021,0.007741,0.992597,0.941779,0.995879,0.854700,0.988644,0.771301
1,014_blend_bias,016_blend_best,0.995881,0.930726,0.987954,0.012046,0.032668,0.033470,0.027174,0.038964,0.697413,0.005494,0.006296,0.994752,0.953273,0.997072,0.872568,0.991863,0.785624
7,016_blend_best,017_realmlp_bias,0.999643,0.993783,0.996309,0.003691,0.033470,0.035542,0.032687,0.036325,0.899866,0.000783,0.002854,0.999556,0.997965,0.999879,0.995564,0.999327,0.998086
4,015_realmlp_shared001,016_blend_best,0.999845,0.997493,0.997939,0.002061,0.034388,0.033470,0.032923,0.034936,0.942390,0.001465,0.000547,0.999807,0.998245,0.999894,0.997662,0.999723,0.998115
5,015_realmlp_shared001,017_realmlp_bias,0.999895,0.997811,0.997825,0.002175,0.034388,0.035542,0.033907,0.036023,0.941244,0.000482,0.001635,0.999877,0.999821,0.999983,0.998623,0.999773,0.999883


,014_blend_bias,015_realmlp_shared001,016_blend_best,017_realmlp_bias,018_xgb_shared003
014_blend_bias,1.000000,0.994167,0.995881,0.993651,0.993720
015_realmlp_shared001,0.994167,1.000000,0.999845,0.999895,0.991146
016_blend_best,0.995881,0.999845,1.000000,0.999643,0.992374
017_realmlp_bias,0.993651,0.999895,0.999643,1.000000,0.990670
018_xgb_shared003,0.993720,0.991146,0.992374,0.990670,1.000000


,014_blend_bias,015_realmlp_shared001,016_blend_best,017_realmlp_bias,018_xgb_shared003
014_blend_bias,1.000000,0.916308,0.930726,0.904324,0.976021
015_realmlp_shared001,0.916308,1.000000,0.997493,0.997811,0.918772
016_blend_best,0.930726,0.997493,1.000000,0.993783,0.929872
017_realmlp_bias,0.904324,0.997811,0.993783,1.000000,0.908650
018_xgb_shared003,0.976021,0.918772,0.929872,0.908650,1.000000


,014_blend_bias,015_realmlp_shared001,016_blend_best,017_realmlp_bias,018_xgb_shared003
014_blend_bias,1.000000,0.985937,0.987954,0.985206,0.984379
015_realmlp_shared001,0.985937,1.000000,0.997939,0.997825,0.982243
016_blend_best,0.987954,0.997939,1.000000,0.996309,0.983381
017_realmlp_bias,0.985206,0.997825,0.996309,1.000000,0.981524
018_xgb_shared003,0.984379,0.982243,0.983381,0.981524,1.000000


,014_blend_bias,015_realmlp_shared001,016_blend_best,017_realmlp_bias,018_xgb_shared003
014_blend_bias,1.000000,0.659451,0.697413,0.649466,0.623540
015_realmlp_shared001,0.659451,1.000000,0.942390,0.941244,0.590848
016_blend_best,0.697413,0.942390,1.000000,0.899866,0.607429
017_realmlp_bias,0.649466,0.941244,0.899866,1.000000,0.582933
018_xgb_shared003,0.623540,0.590848,0.607429,0.582933,1.000000


In [6]:
blend_rows = {}

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row

for set_name, keys in BLEND_SETS.items():
    print(f"===== {set_name}: {keys} =====")
    probas = [oof[k] for k in keys]
    pred_probas = [pred[k] for k in keys]

    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred, weights=np.ones(len(keys)) / len(keys))

    if len(keys) >= 2:
        hc_w, hc_oof, hc_score, hc_hist = hill_climb_weights(y, probas)
        hc_pred = normalize_rows(sum(hc_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=hc_w, extra={"hc_history_len": len(hc_hist)})

        nm_w, nm_oof, nm_score, nm_res = nm_optimize_weights(y, probas)
        nm_pred = normalize_rows(sum(nm_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=nm_w, extra={"nm_success": bool(nm_res.success), "nm_message": str(nm_res.message)})

    # Diagnostic only: in-sample meta screening, not safe final OOF.
    for mode in ["raw", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, penalty="l2", solver="lbfgs", max_iter=2000, random_state=SEED, n_jobs=-1)
            lr.fit(X_meta, y)
            lr_oof = lr.predict_proba(X_meta)
            lr_pred = lr.predict_proba(X_test_meta)
            record_blend(set_name, f"logreg_{mode}", keys, lr_oof, lr_pred, extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            rc_oof = softmax_np(rc.decision_function(X_meta), axis=1)
            rc_pred = softmax_np(rc.decision_function(X_test_meta), axis=1)
            record_blend(set_name, f"ridgeclf_{mode}", keys, rc_oof, rc_pred, extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            rr_oof = normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None))
            rr_pred = normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None))
            record_blend(set_name, f"ridge_reg_{mode}", keys, rr_oof, rr_pred, extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            en_oof_list, en_test_list = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                en_oof_list.append(en.predict(X_meta))
                en_test_list.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(en_oof_list).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(en_test_list).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False)
display(blend_df.head(150))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

===== A_017_only: ['017_realmlp_bias'] =====
[WARN] logreg failed: A_017_only logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_017_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] ridge_reg failed: A_017_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] elasticnet_reg failed: A_017_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: A_017_only raw_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_017_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] ridge_reg failed: A_017_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] elasticnet_reg failed: A_017_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: A_017_only raw_rank

,set_name,method,keys,n_models,balanced_accuracy,weights_json,recall_GALAXY,recall_QSO,recall_STAR,C,risk,alpha,l1_ratio,hc_history_len,nm_success,nm_message
30,F_014_017_018,hc_nonnegative_raw,"014_blend_bias,017_realmlp_bias,018_xgb_shared003",3,0.968872,"{""014_blend_bias"": 0.19335207956110856, ""017_r...",0.965479,0.974441,0.966696,NaN,NaN,NaN,NaN,7.0,NaN,NaN
16,D_017_018,hc_nonnegative_raw,"017_realmlp_bias,018_xgb_shared003",2,0.968861,"{""017_realmlp_bias"": 0.7052217729557467, ""018_...",0.963209,0.975372,0.968002,NaN,NaN,NaN,NaN,7.0,NaN,NaN
58,J_014_015_017_018,hc_nonnegative_raw,"014_blend_bias,015_realmlp_shared001,017_realm...",4,0.968844,"{""014_blend_bias"": 0.15250454053143744, ""015_r...",0.965087,0.974979,0.966467,NaN,NaN,NaN,NaN,10.0,NaN,NaN
51,I_016_017_018,hc_nonnegative_raw,"016_blend_best,017_realmlp_bias,018_xgb_shared003",3,0.968803,"{""016_blend_best"": 0.11141978578535194, ""017_r...",0.963942,0.975201,0.967265,NaN,NaN,NaN,NaN,7.0,NaN,NaN
65,K_014_015_016_017_018,hc_nonnegative_raw,"014_blend_bias,015_realmlp_shared001,016_blend...",5,0.968801,"{""014_blend_bias"": 0.20965998305490563, ""015_r...",0.965630,0.974706,0.966068,NaN,NaN,NaN,NaN,7.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,K_014_015_016_017_018,ridge_reg_raw,"014_blend_bias,015_realmlp_shared001,016_blend...",5,0.965193,None,0.975305,0.972495,0.947778,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN
13,C_018_only,ridge_reg_raw,018_xgb_shared003,1,0.964806,None,0.968340,0.971795,0.954282,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN
12,C_018_only,ridgeclf_raw,018_xgb_shared003,1,0.964806,None,0.968340,0.971795,0.954282,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN
14,C_018_only,elasticnet_reg_raw,018_xgb_shared003,1,0.964785,None,0.968369,0.971778,0.954209,NaN,in_sample_meta_screening,0.0005,0.05,NaN,NaN,NaN


In [7]:
safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy()
safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)

best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
elif best_method in ["hc_nonnegative_raw", "nm_softmax_raw"]:
    probas = [oof[k] for k in best_keys]
    pred_probas = [pred[k] for k in best_keys]
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * probas[i] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred_probas[i] for i in range(len(best_keys))))
else:
    raise RuntimeError(best_method)

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy"
best_pred_npy = OUTDIR / "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy"

np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: best_labels,
})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / "submission_exp_20260607_019_blend_add018_xgb_check_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)

Best safe blend:
{
  "set_name": "F_014_017_018",
  "method": "hc_nonnegative_raw",
  "keys": "014_blend_bias,017_realmlp_bias,018_xgb_shared003",
  "n_models": 3,
  "balanced_accuracy": 0.968872315017554,
  "weights_json": "{\"014_blend_bias\": 0.19335207956110856, \"017_realmlp_bias\": 0.5031124737181557, \"018_xgb_shared003\": 0.3035354467207357}",
  "recall_GALAXY": 0.9654789657730211,
  "recall_QSO": 0.9744414945835432,
  "recall_STAR": 0.9666964846960978,
  "C": NaN,
  "risk": NaN,
  "alpha": NaN,
  "l1_ratio": NaN,
  "hc_history_len": 7.0,
  "nm_success": NaN,
  "nm_message": NaN
}
best_oof_score: 0.968872315017554


,set_name,method,balanced_accuracy,submission,oof_proba,pred_proba
0,F_014_017_018,hc_nonnegative_raw,0.968872,submission_exp_20260607_019_blend_add018_xgb_c...,oof_exp_20260607_019_blend_add018_xgb_check_be...,pred_exp_20260607_019_blend_add018_xgb_check_b...


In [8]:
def score_of(key):
    return float(individual_df.loc[individual_df["key"] == key, "recomputed_oof_cv"].iloc[0])

judgement = {
    "reference_scores": {
        "017_cv": 0.9683022653955233,
        "017_public_lb": 0.96985,
        "015_cv": 0.9681693449222352,
        "015_public_lb": 0.96977,
        "016_cv": 0.968246781866073,
        "016_public_lb": 0.96948,
        "018_cv": 0.965207986418628,
        "018_public_lb": 0.96578,
    },
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "main_questions": {
        "does_018_add_to_017": "Check D_017_018 safe methods and weights.",
        "does_018_add_to_015": "Check E_015_018.",
        "does_018_add_to_014_017": "Check F_014_017_018.",
        "does_018_add_to_016": "Check H/I.",
        "drop_or_keep_018": "If XGB weight is 0 or Public drops, move 018 to drop/low-priority reference.",
    },
    "initial_policy": {
        "017_realmlp_bias": "current_best_reference",
        "015_realmlp_shared001": "stable_backup",
        "016_blend_best": "hold_cv_up_lb_down",
        "018_xgb_shared003": "candidate_under_test",
        "linear_meta_rows": "screening_only_not_final_without_nested_meta",
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")
print(json.dumps(judgement, indent=2, ensure_ascii=False))

{
  "reference_scores": {
    "017_cv": 0.9683022653955233,
    "017_public_lb": 0.96985,
    "015_cv": 0.9681693449222352,
    "015_public_lb": 0.96977,
    "016_cv": 0.968246781866073,
    "016_public_lb": 0.96948,
    "018_cv": 0.965207986418628,
    "018_public_lb": 0.96578
  },
  "individual_best": {
    "key": "017_realmlp_bias",
    "cv": 0.9683022653955233,
    "public_lb": 0.96985
  },
  "best_safe_blend": {
    "exp_id": "exp_20260607_019_blend_add018_xgb_check",
    "best_set_name": "F_014_017_018",
    "best_method": "hc_nonnegative_raw",
    "best_keys": [
      "014_blend_bias",
      "017_realmlp_bias",
      "018_xgb_shared003"
    ],
    "cv_score": 0.968872315017554,
    "weights_json": "{\"014_blend_bias\": 0.19335207956110856, \"017_realmlp_bias\": 0.5031124737181557, \"018_xgb_shared003\": 0.3035354467207357}",
    "oof_path": "/kaggle/working/exp_20260607_019_blend_add018_xgb_check/oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
    "pred_path":

In [9]:
summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Add 018 XGB to current 017 RealMLP branch and evaluate correlation/blend",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add018_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 018 shared003 XGB",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-07",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from the fixed npy-files Kaggle dataset folder, "
            "add 018 shared003 XGB as-is to the current 017 RealMLP branch, "
            "and decide whether XGB adds complementary signal. "
            "Save only the best safe blend OOF/pred proba for the next experiment."
        ),
        "success_criteria": [
            "load 014/015/016/017/018 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations",
            "evaluate predefined blend sets centered on 017/018",
            "separate safe blends from in-sample meta screening",
            "save only best safe blend OOF/pred npy",
            "save best safe blend submission",
            "save memo and summary",
        ],
    },
    "inputs": {
        "npy_dir": str(NPY_DIR),
        "models": resolved_specs,
        "blend_sets": BLEND_SETS,
    },
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "corr_matrix_pearson_flat_proba": "corr_matrix_pearson_flat_proba.csv",
        "corr_matrix_spearman_flat_proba": "corr_matrix_spearman_flat_proba.csv",
        "matrix_argmax_agreement": "matrix_argmax_agreement.csv",
        "matrix_error_jaccard": "matrix_error_jaccard.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "best_blend_info": "best_blend_info.json",
        "saved_submission_candidates": "saved_submission_candidates.csv",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "role_judgement": "role_judgement.json",
        "summary": "blend_add018_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "018 XGB is much weaker as a single model than 017 RealMLP. "
            "The only reason to keep it is if it adds complementary signal in safe blend. "
            "In-sample meta rows are diagnostic only."
        ),
        "next_action": [
            "Review individual_scores.csv",
            "Review pairwise_oof_correlation.csv, especially 017 vs 018",
            "Review safe rows in blend_diagnostics.csv",
            "Submit best safe blend only if it improves over 017 enough",
            "If best safe blend does not improve Public over 017, drop 018 from active candidates",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved files:
 - best_blend_info.json
 - blend_add018_summary.json
 - blend_diagnostics.csv
 - corr_matrix_pearson_flat_proba.csv
 - corr_matrix_spearman_flat_proba.csv
 - individual_scores.csv
 - matrix_argmax_agreement.csv
 - matrix_error_jaccard.csv
 - memo.yml
 - oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy
 - role_judgement.json
 - saved_submission_candidates.csv
 - submission_exp_20260607_019_blend_add018_xgb_check_best_blend.csv
